# Module 1: Quick Start

欢迎来到 Kafka Python 教程的第一章！

本章目标：
- 理解 Kafka 的基本概念：Broker、Topic、Producer、Consumer
- 启动本地 3 节点 KRaft Kafka 集群
- 用 `aiokafka` 写出第一个 Producer 和 Consumer

---

## 前置条件

1. 已安装 Docker & Docker Compose
2. 已在项目根目录执行 `docker compose up -d`，集群正在后台运行
3. 已安装 Python 依赖：`uv sync` 或 `pip install aiokafka`

集群启动后，各 Broker 对本机暴露的端口：

| Broker | 内部地址 | 外部地址（本机访问） |
|--------|---------|--------------------|
| kafka-1 | kafka-1:9092 | localhost:19094 |
| kafka-2 | kafka-2:9092 | localhost:29094 |
| kafka-3 | kafka-3:9092 | localhost:39094 |

## 核心概念速览

```mermaid
graph LR
    P["Producer"] -->|"send_and_wait()"| B["Kafka Broker Topic: my-topic Partition 0 / Partition 1"]
    B -->|"poll / getmany()"| C["Consumer"]
```

- **Broker**：Kafka 服务节点，存储消息并处理读写请求
- **Topic**：消息的逻辑分类，类似数据库的「表」
- **Partition**：Topic 的物理分片，实现并行写入；每条消息有唯一 offset
- **Producer**：向 Topic 发布（写入）消息的客户端
- **Consumer**：从 Topic 订阅（读取）消息的客户端
- **Consumer Group**：多个 Consumer 协同消费同一 Topic，每个 Partition 只分配给组内一个 Consumer

## 0. 环境配置

In [1]:
import asyncio
import logging
from aiokafka import AIOKafkaProducer, AIOKafkaConsumer
from aiokafka.admin import AIOKafkaAdminClient, NewTopic

# aiokafka 在集群启动初期的瞬态重试（元数据未就绪、coordinator 初始化）
# 会以 WARNING 级别打印日志，但操作本身会自动重试成功。
# 对学习 Kafka 概念无意义，统一屏蔽至 ERROR 级别。
logging.getLogger("aiokafka").setLevel(logging.ERROR)

BROKERS = "localhost:19094,localhost:29094,localhost:39094"
TOPIC = "quickstart-topic"
GROUP = "quickstart-group"

print(f"Brokers: {BROKERS}")
print(f"Topic:   {TOPIC}")
print(f"Group:   {GROUP}")

Brokers: localhost:19094,localhost:29094,localhost:39094
Topic:   quickstart-topic
Group:   quickstart-group


## 1. 检查 Kafka 集群是否就绪

In [2]:
async def wait_for_kafka(brokers: str, retries: int = 20, delay: float = 3.0) -> bool:
    """等待 Kafka 集群完全就绪，包括 Broker 连通性和 Consumer Group 协调器。"""
    print("正在检测 Kafka 集群...")
    for i in range(retries):
        try:
            producer = AIOKafkaProducer(bootstrap_servers=brokers)
            await producer.start()
            await producer.stop()
            # 同时验证 Consumer Group 协调器（触发 __consumer_offsets 初始化）
            consumer = AIOKafkaConsumer(
                bootstrap_servers=brokers,
                group_id="__kafka_healthcheck__",
                enable_auto_commit=False,
            )
            await consumer.start()
            await consumer.stop()
            print("✓ Kafka 集群已就绪！")
            return True
        except Exception:
            print(f"  等待中... ({i+1}/{retries})")
            await asyncio.sleep(delay)
    print("✗ 连接超时，请确认 docker compose up -d 已执行")
    return False

await wait_for_kafka(BROKERS)

正在检测 Kafka 集群...
✓ Kafka 集群已就绪！


True

## 2. 创建 Topic

Kafka 会在 Producer 首次发送时自动创建 Topic（默认配置下），但显式创建可以精确控制 **分区数** 和 **副本数**。

本章先用 1 个分区、3 个副本（与 Broker 数量相同），后续章节会深入讲解分区与副本。

In [3]:
async def create_topic(brokers: str, topic: str, num_partitions: int = 1, replication_factor: int = 3):
    admin = AIOKafkaAdminClient(bootstrap_servers=brokers)
    await admin.start()
    try:
        existing = await admin.list_topics()
        if topic in existing:
            print(f"Topic '{topic}' 已存在，跳过创建")
            return
        await admin.create_topics([
            NewTopic(name=topic, num_partitions=num_partitions, replication_factor=replication_factor)
        ])
        print(f"✓ Topic '{topic}' 创建成功（分区={num_partitions}, 副本={replication_factor}）")
    finally:
        await admin.close()

await create_topic(BROKERS, TOPIC)

Topic 'quickstart-topic' 已存在，跳过创建


## 3. Producer：发送消息

`AIOKafkaProducer` 是异步 Producer，核心 API：

| 方法 | 说明 |
|------|------|
| `await producer.start()` | 建立连接，获取集群元数据 |
| `await producer.send_and_wait(topic, value)` | 发送消息并等待 Broker 确认 |
| `await producer.stop()` | 刷新缓冲区并关闭连接 |

In [4]:
async def produce(brokers: str, topic: str, count: int = 5):
    producer = AIOKafkaProducer(bootstrap_servers=brokers)
    await producer.start()
    try:
        for i in range(1, count + 1):
            message = f"Hello Kafka! #{i}"
            record_metadata = await producer.send_and_wait(topic, message.encode())
            print(
                f"[Producer] 发送: {message!r}"
                f" -> partition={record_metadata.partition}, offset={record_metadata.offset}"
            )
    finally:
        await producer.stop()
    print("[Producer] 完成")

await produce(BROKERS, TOPIC)

[Producer] 发送: 'Hello Kafka! #1' -> partition=0, offset=5
[Producer] 发送: 'Hello Kafka! #2' -> partition=0, offset=6
[Producer] 发送: 'Hello Kafka! #3' -> partition=0, offset=7
[Producer] 发送: 'Hello Kafka! #4' -> partition=0, offset=8
[Producer] 发送: 'Hello Kafka! #5' -> partition=0, offset=9
[Producer] 完成


## 4. Consumer：接收消息

`AIOKafkaConsumer` 是异步 Consumer，关键参数：

| 参数 | 说明 |
|------|------|
| `group_id` | Consumer Group ID；同一组内的 Consumer 不重复消费 |
| `auto_offset_reset` | 无已提交 offset 时的起始位置：`earliest`（最早）或 `latest`（最新）|
| `enable_auto_commit` | 自动提交 offset（默认 True），生产场景建议关闭手动控制 |

In [5]:
async def consume(brokers: str, topic: str, group: str, max_messages: int = 5):
    consumer = AIOKafkaConsumer(
        topic,
        bootstrap_servers=brokers,
        group_id=group,
        auto_offset_reset="earliest",
    )
    await consumer.start()
    try:
        count = 0
        async for msg in consumer:
            print(
                f"[Consumer] 收到: {msg.value.decode()!r}"
                f" <- partition={msg.partition}, offset={msg.offset}"
            )
            count += 1
            if count >= max_messages:
                break
    finally:
        await consumer.stop()
    print("[Consumer] 完成")

await consume(BROKERS, TOPIC, GROUP)

[Consumer] 收到: 'Hello Kafka! #1' <- partition=0, offset=5
[Consumer] 收到: 'Hello Kafka! #2' <- partition=0, offset=6
[Consumer] 收到: 'Hello Kafka! #3' <- partition=0, offset=7
[Consumer] 收到: 'Hello Kafka! #4' <- partition=0, offset=8
[Consumer] 收到: 'Hello Kafka! #5' <- partition=0, offset=9
[Consumer] 完成


## 5. 完整 Demo：同时运行 Producer 和 Consumer

实际场景中 Producer 和 Consumer 是独立进程。这里用 `asyncio.gather` 并发运行两者，模拟消息的实时流转。

In [ ]:
import time

STREAM_TOPIC = "quickstart-stream"

async def stream_producer(brokers: str, topic: str, ready: asyncio.Event, count: int = 5):
    producer = AIOKafkaProducer(bootstrap_servers=brokers)
    await producer.start()
    try:
        # 等 consumer 通知"已定位到 latest"，再开始发送
        await ready.wait()
        for i in range(1, count + 1):
            msg = f"event-{i} @ {time.strftime('%H:%M:%S')}"
            await producer.send_and_wait(topic, msg.encode())
            print(f"  [P] → {msg}")
            await asyncio.sleep(0.5)
    finally:
        await producer.stop()

async def stream_consumer(brokers: str, topic: str, group: str, ready: asyncio.Event, count: int = 5):
    consumer = AIOKafkaConsumer(
        topic,
        bootstrap_servers=brokers,
        group_id=group,
        auto_offset_reset="latest",
    )
    await consumer.start()
    try:
        # consumer.start() 只完成 group join，"latest" offset 在第一次 fetch 时才确定。
        # 用 getmany 触发一次 fetch，确保已定位到 latest，再通知 producer 开始发送。
        await consumer.getmany(timeout_ms=500)
        ready.set()

        received = 0
        async for msg in consumer:
            print(f"  [C] ← {msg.value.decode()} (offset={msg.offset})")
            received += 1
            if received >= count:
                break
    finally:
        await consumer.stop()

await create_topic(BROKERS, STREAM_TOPIC)

print("=== 实时消息流演示 ===")
ready = asyncio.Event()
await asyncio.gather(
    stream_producer(BROKERS, STREAM_TOPIC, ready, count=5),
    stream_consumer(BROKERS, STREAM_TOPIC, "stream-group", ready, count=5),
)
print("=== 演示完成 ===")

## 6. 观察 Offset 的累积

每次 Producer 写入，offset 单调递增。重新运行 Producer 后再用 Consumer 以 `earliest` 重置，会从头读取所有历史消息。

**尝试以下操作：**
1. 再次运行「3. Producer」单元格（追加 5 条消息）
2. 重新运行「4. Consumer」单元格——注意已提交 offset，Consumer 不会重复读已消费部分
3. 修改 `group_id` 为全新值，再运行 Consumer——会从 earliest 读取全部消息

In [7]:
# 用新 group 从头读取所有消息（使用 getmany 避免 consumer_timeout_ms 兼容性问题）
async def consume_all(brokers: str, topic: str, group: str):
    consumer = AIOKafkaConsumer(
        topic,
        bootstrap_servers=brokers,
        group_id=group,
        auto_offset_reset="earliest",
    )
    await consumer.start()
    count = 0
    try:
        while True:
            results = await consumer.getmany(timeout_ms=1500, max_records=50)
            if not results:
                break
            for tp_msgs in results.values():
                for msg in tp_msgs:
                    print(f"  offset={msg.offset:3d} | {msg.value.decode()}")
                    count += 1
    finally:
        await consumer.stop()
    print(f"共读取 {count} 条消息")

await consume_all(BROKERS, TOPIC, "inspect-group-1")

  offset=  0 | Hello Kafka! #1
  offset=  1 | Hello Kafka! #2
  offset=  2 | Hello Kafka! #3
  offset=  3 | Hello Kafka! #4
  offset=  4 | Hello Kafka! #5
  offset=  5 | Hello Kafka! #1
  offset=  6 | Hello Kafka! #2
  offset=  7 | Hello Kafka! #3
  offset=  8 | Hello Kafka! #4
  offset=  9 | Hello Kafka! #5
共读取 10 条消息


## 总结

| 概念 | 要点 |
|------|------|
| Broker | 存储消息的服务节点，多个 Broker 组成集群 |
| Topic | 消息的逻辑命名空间，由一个或多个 Partition 组成 |
| Partition | Topic 的物理分片，消息在分区内有序且 offset 单调递增 |
| Producer | 向 Broker 发送消息；`send_and_wait` 等待 Broker 确认 |
| Consumer | 从 Broker 拉取消息；`group_id` 决定消费进度的共享粒度 |
| Offset | 分区内消息的唯一序号；Consumer 通过提交 offset 记录消费进度 |

---

## 下一章

**Module 2: Topics & Partitions** — 深入理解分区如何实现并行写入和消费，以及消息键（key）如何决定分区路由。